<a href="https://colab.research.google.com/github/MusicalManiac/SatelliteDataAI-UOA/blob/main/Lab4_Answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Code to find answer to Question 2
def count_samples(files):
    total = 0
    for f in files:
        total += np.load(f)["X"].shape[0]
    return total

n_train = count_samples(train_files)
n_val   = count_samples(val_files)

print("Training patches:", n_train)
print("Validation patches:", n_val)

In [ ]:
# Code to Question 6
import ee
import matplotlib.pyplot as plt
import rasterio
import numpy as np
import glob
from google.colab import drive
ee.Authenticate()
ee.Initialize(project='earthengine-ml-testing-504003')

# Negative-class sampling locations: land cover types with NO solar panels expected
negative_sites = {
    'urban_roof_hard_negative': ee.Geometry.Point(174.7633, -36.8485),
    'farmland': ee.Geometry.Point(174.9000, -37.0500),
    'forest': ee.Geometry.Point(174.5200, -36.9200),
    'water': ee.Geometry.Point(174.8300, -36.7900),
}

PATCH_SIZE_M = 640  # matching existing patch footprint in metres (128px * 10m if native S2 res... adjust to match Solafune patch size)

def export_negative_patch(name, point):
    region = point.buffer(PATCH_SIZE_M / 2).bounds()
    img = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(region)
           .filterDate('2023-01-01', '2023-06-01')
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
           .median()
           .select(['B4', 'B3', 'B2', 'B8', 'B11']))  # RGB + NIR + SWIR
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=f'negative_{name}',
        folder='solar_negatives',
        region=region,
        scale=10,
        fileFormat='GeoTIFF'
    )
    task.start()
    print(f"Export started: {name}")

for name, pt in negative_sites.items():
    export_negative_patch(name, pt)


drive.mount('/content/drive', force_remount=True)
drive_folder = '/content/drive/MyDrive/solar_negatives'
negative_paths = sorted(glob.glob(f'{drive_folder}/*.tif'))
print(negative_paths)

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

def load_rgb(path, band_indices=(1,2,3)):
    with rasterio.open(path) as src:
        img = src.read(list(band_indices)).astype(np.float32)
        img = np.transpose(img, (1,2,0))
        p2, p98 = np.percentile(img, (2, 98))
        img = np.clip((img - p2) / (p98 - p2 + 1e-6), 0, 1)
    return img

positive_paths = [pairs[0][0], pairs[1][0], pairs[2][0]]
negative_paths_subset = negative_paths[:3]

fig, axes = plt.subplots(2, 3, figsize=(9, 6), dpi=300)

for i, p in enumerate(positive_paths):
    axes[0, i].imshow(load_rgb(p, band_indices=(4,3,2)))   # 12-band Solafune order: B4,B3,B2
    axes[0, i].set_title('Existing: Panel present', fontsize=10)
    axes[0, i].axis('off')

for i, p in enumerate(negative_paths_subset):
    axes[1, i].imshow(load_rgb(p, band_indices=(1,2,3)))   # 5-band EE export order: already B4,B3,B2
    axes[1, i].set_title('New: No panel (negative)', fontsize=10)
    axes[1, i].axis('off')

caption = (
    "Figure 1. Comparison of existing training patches (top, all containing "
    "solar panels) against newly added negative patches (bottom, drawn from "
    "urban, farmland, and forest land cover with no panels present), added "
    "to correct the missing-negative-class flaw identified in Q5."
)
fig.text(0.5, -0.03, caption, ha='center', va='top', wrap=True, fontsize=9)
plt.tight_layout()
plt.savefig('q6_positive_vs_negative_patches.png', dpi=300, bbox_inches='tight')
plt.savefig('q6_positive_vs_negative_patches.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import rasterio
import numpy as np
from pathlib import Path

# Create all-zero masks matching each negative patch's dimensions
negative_mask_dir = Path("/content/negative_masks")
negative_mask_dir.mkdir(exist_ok=True)

negative_pairs = []
for neg_path in negative_paths:
    with rasterio.open(neg_path) as src:
        profile = src.profile
        h, w = src.height, src.width

    zero_mask = np.zeros((h, w), dtype=np.uint8)
    mask_path = negative_mask_dir / (Path(neg_path).stem + "_mask.tif")

    mask_profile = profile.copy()
    mask_profile.update(count=1, dtype='uint8')
    with rasterio.open(mask_path, 'w', **mask_profile) as dst:
        dst.write(zero_mask, 1)

    negative_pairs.append((Path(neg_path), mask_path))

print(f"Created {len(negative_pairs)} negative pairs")

# Merge with existing positive pairs
all_pairs = pairs + negative_pairs
print(f"Total pairs now: {len(all_pairs)} ({len(pairs)} positive + {len(negative_pairs)} negative)")

OUTPUT_DIR_V2 = "patch_store_v2"
os.makedirs(OUTPUT_DIR_V2, exist_ok=True)

TARGET_HEIGHT, TARGET_WIDTH = 256, 256
BATCH_SIZE = 100
COMMON_BANDS = 5

X_batch, Y_batch = [], []
batch_idx = 0

for i, (s2_path, mask_path) in enumerate(tqdm(all_pairs, desc="Processing")):
    with rasterio.open(s2_path) as src:
        img = src.read().astype(np.float32)
        img = np.transpose(img, (1, 2, 0))

    # Slice to common band count BEFORE resizing/padding
    img = img[..., :COMMON_BANDS]

    with rasterio.open(mask_path) as src:
        mask = src.read(1).astype(np.uint8)

    img = resize_or_pad(img, TARGET_HEIGHT, TARGET_WIDTH)
    mask = resize_or_pad(mask, TARGET_HEIGHT, TARGET_WIDTH)
    img = img / 10000.0

    X_batch.append(img)
    Y_batch.append(mask)

    if len(X_batch) >= BATCH_SIZE:
        X_batch = np.stack(X_batch)
        Y_batch = np.stack(Y_batch)
        np.savez_compressed(os.path.join(OUTPUT_DIR_V2, f"batch_{batch_idx:04d}.npz"), X=X_batch, Y=Y_batch)
        batch_idx += 1
        X_batch, Y_batch = [], []

if len(X_batch) > 0:
    X_batch = np.stack(X_batch)
    Y_batch = np.stack(Y_batch)
    np.savez_compressed(os.path.join(OUTPUT_DIR_V2, f"batch_{batch_idx:04d}.npz"), X=X_batch, Y=Y_batch)

print(f"Done. Total batches written: {batch_idx + (1 if len(X_batch) > 0 else 0)}")

TARGET_HEIGHT, TARGET_WIDTH = 128, 128
BANDS = 5          # increased from 3 → now includes NIR + SWIR, not just RGB
BATCH_SIZE = 8     # increased from 4, since we now have more diverse data to batch
MAX_TRAIN_FILES = 6   # slightly more files since dataset grew with negatives
MAX_VAL_FILES = 2

all_files_v2 = sorted(glob.glob("patch_store_v2/*.npz"))
np.random.shuffle(all_files_v2)
train_files = all_files_v2[:MAX_TRAIN_FILES]
val_files = all_files_v2[-MAX_VAL_FILES:]

# Mask to binary label (unchanged from original)
def mask_to_label(mask_batch):
    """1 if any solar pixel in mask, else 0"""
    labels = (np.sum(mask_batch, axis=(1,2,3)) > 0).astype(np.float32)
    return labels


# Data loader (updated to use BANDS=5)
def npz_loader_cls(path):
    data = np.load(path.numpy().decode("utf-8"))
    X = data["X"]
    Y_mask = data["Y"]

    X = X[..., :BANDS]

    from skimage.transform import resize
    X_resized = np.zeros((X.shape[0], TARGET_HEIGHT, TARGET_WIDTH, BANDS), dtype=np.float32)
    for i in range(X.shape[0]):
        X_resized[i] = resize(X[i], (TARGET_HEIGHT, TARGET_WIDTH, BANDS), anti_aliasing=True)

    X = X_resized
    Y = mask_to_label(Y_mask)
    return X, Y


# TensorFlow dataset pipeline (unchanged)
def tf_wrapper(path):
    X, Y = tf.py_function(npz_loader_cls, [path], [tf.float32, tf.float32])
    X.set_shape([None, TARGET_HEIGHT, TARGET_WIDTH, BANDS])
    Y.set_shape([None])
    return tf.data.Dataset.from_tensor_slices((X, Y))

def make_cls_dataset(file_list, batch_size=BATCH_SIZE, shuffle=True, repeat=True):
    files = tf.data.Dataset.from_tensor_slices(file_list)
    if shuffle:
        files = files.shuffle(len(file_list))

    ds = files.interleave(
        lambda f: tf_wrapper(f),
        cycle_length=4,
        num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(2048)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    if repeat:
        ds = ds.repeat()

    return ds

train_ds = make_cls_dataset(train_files, batch_size=BATCH_SIZE, shuffle=True)
val_ds   = make_cls_dataset(val_files, batch_size=BATCH_SIZE, shuffle=False, repeat=False)

# Model definition (same architecture, now takes 5-band input)
for X_sample, _ in train_ds.take(1):
    input_shape = X_sample.shape[1:]
    break

print("Input shape:", input_shape)

inputs = layers.Input(shape=input_shape)
x = layers.Conv2D(16, 3, activation='relu', padding='same')(inputs)
x = layers.MaxPooling2D((2,2))(x)
x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(32, activation='relu')(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model_v2 = models.Model(inputs, outputs)
model_v2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_v2.summary()


# Count samples and set steps
def count_samples(files):
    total = 0
    for f in files:
        total += np.load(f)["X"].shape[0]
    return total

n_train = count_samples(train_files)
n_val   = count_samples(val_files)

steps_per_epoch = max(1, n_train // BATCH_SIZE)
validation_steps = max(1, n_val // BATCH_SIZE)

print(f"Training samples: {n_train} | Validation samples: {n_val}")
print(f"Steps per epoch: {steps_per_epoch} | Validation steps: {validation_steps}")


# Train
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )]


history_v2 = model_v2.fit(
    train_ds,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_ds,
    validation_steps=validation_steps,
    epochs=10,   # increased from 5 since the task is now genuinely harder
    callbacks=callbacks)

# Plot training curves
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history_v2.history['loss'], label="train")
plt.plot(history_v2.history['val_loss'], label="val")
plt.title("Loss (v2: with negatives)")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history_v2.history['accuracy'], label="train")
plt.plot(history_v2.history['val_accuracy'], label="val")
plt.title("Accuracy (v2: with negatives)")
plt.legend()
plt.show()

# Prediction grid - should now show a MIX of true 0s and 1s
def show_patch_predictions_grid(dataset, model, n_rows=2, n_cols=5):
    def stretch(img):
        img = img.astype(np.float32)
        p2, p98 = np.percentile(img, (2, 98))
        img = np.clip((img - p2) / (p98 - p2 + 1e-6), 0, 1)
        return img

    for X, Y_true in dataset.take(1):
        Y_pred = model.predict(X, verbose=0)
        n_total = min(n_rows * n_cols, X.shape[0])

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*3, n_rows*3))
        axes = axes.flatten()

        for i in range(n_total):
            rgb = X[i, :, :, :3].numpy()
            rgb = stretch(rgb)
            axes[i].imshow(rgb)
            axes[i].set_title(f"T:{int(Y_true[i].numpy())}\nP:{Y_pred[i,0]:.2f}")
            axes[i].axis("off")

        for i in range(n_total, len(axes)):
            axes[i].axis("off")

        plt.tight_layout()
        plt.show()

show_patch_predictions_grid(val_ds, model_v2, n_rows=1, n_cols=5)